<a href="https://colab.research.google.com/github/argg1006/TRABAJOS/blob/main/04_erpscm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
base_datos_erp = pd.read_csv('inventario_ERPSCM.csv')
ventas_totales_dia = 0.0

# parte 1: Ventas e inventario

def procesar_venta(id_producto, cantidad_deseada):
    global ventas_totales_dia
    producto_encontrado = False

    print(f"Se desea vender {cantidad_deseada} unidad(es) del producto {id_producto}...")

    for indice, producto in base_datos_erp.iterrows():
        if producto['id'] == id_producto:
            producto_encontrado = True

            # Verifica si hay suficiente stock
            if producto['stock_actual'] >= cantidad_deseada:

                # Actualiza el stock
                base_datos_erp.at[indice, 'stock_actual'] = (
                    producto['stock_actual'] - cantidad_deseada
                )

                # Calcula el ingreso
                ingreso = producto['precio'] * cantidad_deseada
                ventas_totales_dia += ingreso

                print(
                    f"   Transacción exitosa: {producto['nombre']}. "
                    f"Nuevo stock: {base_datos_erp.at[indice, 'stock_actual']}"
                )
            else:
                print(f"   Error: Stock insuficiente para {producto['nombre']}.")

    if not producto_encontrado:
        print("   Error: Producto no encontrado.")

# Parte 2: SCM PUSH

def ejecutar_modelo_push():
    print("\nEJECUTANDO MODELO SCM PUSH (MAKE-TO-STOCK)")
    print("Regla: Comprar anticipadamente si Stock Actual < Pronóstico.")

    compras_realizadas = 0

    for indice, producto in base_datos_erp.iterrows():
        stock = producto['stock_actual']
        pronostico = producto['pronostico_venta']

        if stock < pronostico:

            # Comprar la diferencia
            cantidad_a_comprar = pronostico - stock

            print(
                f"   PUSH: {producto['nombre']} "
                f"(Stock: {stock} | Pronóstico: {pronostico})"
            )
            print(
                f"   Generando orden de compra por "
                f"{cantidad_a_comprar} unidades."
            )

            # Simular llegada de mercadería
            base_datos_erp.at[indice, 'stock_actual'] += cantidad_a_comprar
            compras_realizadas += 1

    if compras_realizadas == 0:
        print("   PUSH Reporte: Inventario suficiente para los pronósticos.")


# Parte 3: SCM
def ejecutar_modelo_pull(id_producto, cantidad_cliente):
    print("\n EJECUTANDO MODELO PULL (MAKE-TO-ORDER)")
    print(f"Cliente solicita {cantidad_cliente} unidades "f"del producto {id_producto}...")

    for indice, producto in base_datos_erp.iterrows():
        if producto['id'] == id_producto:
            stock = producto['stock_actual']

            if stock >= cantidad_cliente:
                print(
f"Stock suficiente ({stock}). "f"Se despacha del inventario existente.")
                base_datos_erp.at[indice, 'stock_actual'] -= cantidad_cliente

            else:
                faltante = cantidad_cliente - stock

                print(f"   Stock insuficiente. Faltan {faltante} unidades.")
                print(f"   -> PULL ACTIVADO: Comprando "f"{faltante} unidades urgentemente...")

                print("Pedido entregado al cliente.")

                # En Pull puro el inventario queda en 0
                base_datos_erp.at[indice, 'stock_actual'] = 0

# Parte 4: Pruebas

print("--- 1. PROBANDO ERP (VENTAS) ---")
procesar_venta(101, 1)      # Debe funcionar
procesar_venta(100, 10)     # Debe fallar (stock insuficiente)

print(f"\nDinero en Caja (Finanzas): ${ventas_totales_dia}")

print("\n--- 2. PROBANDO SCM (PUSH) ---")
ejecutar_modelo_push()

print("\n--- 3. PROBANDO SCM (PULL) ---")
ejecutar_modelo_pull(105, 200)

print("\n ESTADO FINAL DEL INVENTARIO (ERP INTEGRADO)")
display(base_datos_erp)

--- 1. PROBANDO ERP (VENTAS) ---
Se desea vender 1 unidad(es) del producto 101...
   Transacción exitosa: Cable HDMI Básico (Stock Alto). Nuevo stock: 199
Se desea vender 10 unidad(es) del producto 100...
   Error: Stock insuficiente para Laptop Gamer Muestra (Bajo Stock).

Dinero en Caja (Finanzas): $15.0

--- 2. PROBANDO SCM (PUSH) ---

EJECUTANDO MODELO SCM PUSH (MAKE-TO-STOCK)
Regla: Comprar anticipadamente si Stock Actual < Pronóstico.
   PUSH: Laptop Gamer Muestra (Bajo Stock) (Stock: 2 | Pronóstico: 25)
   Generando orden de compra por 23 unidades.
   PUSH: Audífonos Noise Cancelling Lenovo Gamer (Stock: 79 | Pronóstico: 88)
   Generando orden de compra por 9 unidades.
   PUSH: Mouse Inalámbrico Xiaomi Slim (Stock: 83 | Pronóstico: 86)
   Generando orden de compra por 3 unidades.
   PUSH: Lavadora Apple Business (Stock: 17 | Pronóstico: 23)
   Generando orden de compra por 6 unidades.
   PUSH: Parlante Bluetooth Canon RGB (Stock: 2 | Pronóstico: 23)
   Generando orden de compra 

,id,nombre,precio,stock_actual,punto_reorden,pronostico_venta
0,100,Laptop Gamer Muestra (Bajo Stock),1500.00,25,10,25
1,101,Cable HDMI Básico (Stock Alto),15.00,199,20,50
2,102,Audífonos Noise Cancelling Lenovo Gamer,306.71,88,26,88
3,103,Mouse Inalámbrico Xiaomi Slim,90.75,86,23,86
4,104,Lavadora Apple Business,585.53,23,26,23
...,...,...,...,...,...,...
297,397,Smartphone Lenovo Pro,1325.42,80,22,80
298,398,Parlante Bluetooth Xiaomi Pro,139.87,71,30,71
299,399,Laptop HP Max,1927.56,71,26,68
300,400,Laptop LG Pro,1284.85,48,18,48
